### Loading the dataset, preprocessing them (resampling) and formating them for the training

In [ ]:
import os
import pandas as pd
import numpy as np
import datasets
from transformers import RobertaTokenizer
from sklearn.utils import resample

In [ ]:
dataset = "/datasets/values_labels"
labels = [ "Self-direction: thought", "Self-direction: action", "Stimulation",  "Hedonism", "Achievement", "Power: dominance", "Power: resources", "Face", "Security: personal", "Security: societal", "Tradition", "Conformity: rules", "Conformity: interpersonal", "Humility", "Benevolence: caring", "Benevolence: dependability", "Universalism: concern", "Universalism: nature", "Universalism: tolerance", "No Value"]
num_labels = len(labels)

model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)

In [ ]:
def create_dataset(directory, output_path, tokenizer, load_labels=True, subset_fraction=1):
    sentences_file_path = os.path.join(directory, "sentences.tsv")
    labels_file_path = os.path.join(directory, f"{dataset}.tsv")
    
    data_frame = pd.read_csv(sentences_file_path, encoding="utf-8", sep="\t", header=0)

    if load_labels and os.path.isfile(labels_file_path):
        labels_frame = pd.read_csv(labels_file_path, encoding="utf-8", sep="\t", header=0)
        labels_frame = pd.merge(data_frame, labels_frame, on=["Text-ID", "Sentence-ID"], how="inner")
        labels_frame = labels_frame[labels_frame[labels].sum(axis=1) > 0]
        
        class_distribution = labels_frame.iloc[:, 2:].sum(axis=0).apply(pd.to_numeric, errors='coerce')
        target_count = 300
        to_oversample = class_distribution[class_distribution < target_count].index
        to_undersample = class_distribution[class_distribution > target_count].index
        
        balanced_data = []
        for column in to_undersample:
            positive_samples = labels_frame[labels_frame[column] == 1]
            
            undersampled = resample(
                positive_samples,
                replace=False, 
                n_samples=target_count, 
            )
            
            balanced_data.append(undersampled)
        for column in to_oversample:
            positive_samples = labels_frame[labels_frame[column] == 1]
            
            oversampled = resample(
                positive_samples,
                replace=True,  
                n_samples=target_count,
            )
            balanced_data.append(oversampled)
        
        df_balanced = pd.concat(balanced_data)
        df_balanced = df_balanced.sample(frac=1).reset_index(drop=True)

        labels_matrix = np.zeros((df_balanced.shape[0], len(labels)))
        for idx, label in enumerate(labels):
            if label in df_balanced.columns:
                labels_matrix[:, idx] = (df_balanced[label] >= 0.5).astype(int)
        
        encoded_sentences = tokenizer(df_balanced["Text"].to_list(), truncation=True)
        
        if load_labels:
            encoded_sentences["labels"] = labels_matrix.tolist()
    
    # Save the dataset to disk
    dataset = datasets.Dataset.from_dict(encoded_sentences)
    
    # Subset selection
    subset_size = int(len(dataset) * subset_fraction)
    dataset = dataset.select(range(subset_size))
    
    dataset.save_to_disk(output_path)
    print(f"Saved preprocessed dataset to {output_path}")
    
# Process datasets
directory_test="../datasets/valueeval24/test-english"
directory_train="../datasets/valueeval24/training-english"
directory_validation="../datasets/valueeval24/validation-english"

create_dataset(directory_train, "../datasets/processed_train", tokenizer)
create_dataset(directory_validation, "../datasets/processed_validation", tokenizer)
create_dataset(directory_test, "../datasets/processed_test", tokenizer)

Saving the dataset (0/1 shards):   0%|          | 0/44758 [00:00<?, ? examples/s]

Saved preprocessed dataset to ../datasets/processed_train


Saving the dataset (0/1 shards):   0%|          | 0/14904 [00:00<?, ? examples/s]

Saved preprocessed dataset to ../datasets/processed_validation


Saving the dataset (0/1 shards):   0%|          | 0/14569 [00:00<?, ? examples/s]

Saved preprocessed dataset to ../datasets/processed_test
